In [1]:
!pip install -q pgmpy ucimlrepo


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

C:\Users\Prajwal\AppData\Roaming\Python\Python314\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [4]:
columns = [
    "age", "sex", "cp", "trestbps", "chol",
    "fbs", "restecg", "thalach", "exang",
    "oldpeak", "slope", "ca", "thal", "num"
]

df = pd.read_csv(
    "processed.cleveland.data",
    names=columns,
    na_values="?"
)

print(df.head())
print("Dataset shape:", df.shape)

    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  num  
0    3.0  0.0   6.0    0  
1    2.0  3.0   3.0    2  
2    2.0  2.0   7.0    1  
3    3.0  0.0   3.0    0  
4    1.0  0.0   3.0    0  
Dataset shape: (303, 14)


In [5]:
for col in ["age", "cp", "thalach", "exang", "num"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=["age", "cp", "thalach", "exang", "num"]
).copy()

df["target"] = (df["num"] > 0).astype(int)

In [6]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 40, 55, 100],
    labels=["young", "middle", "senior"]
)

df["thalach_group"] = pd.cut(
    df["thalach"],
    bins=[0, 120, 160, 250],
    labels=["low", "medium", "high"]
)

df["cp"] = df["cp"].astype(int).astype(str)
df["exang"] = df["exang"].astype(int).astype(str)

data = df[
    ["age_group", "cp", "exang", "thalach_group", "target"]
].dropna()

data["age_group"] = data["age_group"].astype(str)
data["thalach_group"] = data["thalach_group"].astype(str)

print(data.head())

  age_group cp exang thalach_group  target
0    senior  1     0        medium       0
1    senior  4     1           low       1
2    senior  4     1        medium       1
3     young  3     0          high       0
4    middle  2     0          high       0


In [10]:
estimator = BayesianEstimator(model, data)

cpds = estimator.get_parameters(
    prior_type="BDeu",
    equivalent_sample_size=5
)

model.add_cpds(*cpds)

print("Model valid:", model.check_model())

Model valid: True


C:\Users\Prajwal\AppData\Local\Temp\ipykernel_16148\968358265.py:1: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(model, data)


In [12]:
result = inference.query(
    variables=["target"],
    evidence={
        "age_group": "middle",
        "cp": "4",
        "exang": "1",
        "thalach_group": "low"
    }
)

print(result)

for state, probability in zip(
    result.state_names["target"],
    result.values
):
    label = "Heart Disease" if state == 1 else "No Heart Disease"
    print(f"{label}: {probability:.3f}")

+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.0367 |
+-----------+---------------+
| target(1) |        0.9633 |
+-----------+---------------+
No Heart Disease: 0.037
Heart Disease: 0.963
